In [1]:
# write the list of necessary packages here:
!pip install pandas
!pip install nltk
!pip install spacy
!pip install scikit-learn
!pip install sklearn-crfsuite


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Training a model on Named Entity Recognition task

Token classification refers to the task of classifying individual tokens in a sentence. One of the most common token
classification tasks is Named Entity Recognition (NER). NER attempts to find a label for each entity in a sentence,
such as a person, location, or organization. In this assignment, you will learn how to train a model on the [CoNLL 2023 NER Dataset](https://www.clips.uantwerpen.be/conll2003/ner/) dataset to detect new entities.

### Loading the dataset

In [2]:
# import your packages here:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import sequence_accuracy_score, flat_classification_report

In [3]:
train_df = pd.read_csv("ner_data/train.txt", header=0, sep=" ")
val_df = pd.read_csv("ner_data/val.txt", header=0, sep=" ")
test_df = pd.read_csv("ner_data/test.txt", header=0, sep=" ")

print(f"{train_df.shape}, {val_df.shape}, {test_df.shape}")

(204566, 4), (51577, 4), (46665, 4)


The CoNLL-2003 shared task data files contain four columns separated by a single space. Each word has been put on a separate line and there is an empty line after each sentence. The first item on each line is a word, the second a part-of-speech (POS) tag, the third a syntactic chunk tag and the fourth the named entity tag. The chunk tags and the named entity tags have the format I-TYPE which means that the word is inside a phrase of type TYPE. Only if two phrases of the same type immediately follow each other, the first word of the second phrase will have tag B-TYPE to show that it starts a new phrase. A word with tag O is not part of a phrase. Here is an example:

In [4]:
train_df.head()

,-DOCSTART-,-X-,-X-.1,O
0,EU,NNP,B-NP,B-ORG
1,rejects,VBZ,B-VP,O
2,German,JJ,B-NP,B-MISC
3,call,NN,I-NP,O
4,to,TO,B-VP,O


In [5]:
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

labels_vocab = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}
labels_vocab_reverse = {v:k for k,v in labels_vocab.items()}

### Feature Extraction
 
You need to extract features for each token. The features can be:
• Basic features: Token itself, token lowercase, prefix/suffix of the token.
• Context features: Neighboring tokens (previous/next token).
• Linguistic features: Part-of-speech (POS) tags or word shapes (capitalization, digits,
etc.).
Note that you are expected to briefly mention which features you employ for training your
model.

In [6]:
# write your code here:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import sequence_accuracy_score, flat_classification_report

def word2features(sent, i):
   
    word = sent[i][0]  # Extract the token

    features = {
        # Basic features
        'word': word,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],  # suffix
        'word[:3]': word[:3],    # prefix
        
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        
        'word.prev': sent[i - 1][0] if i > 0 else '<START>',
        'word.next': sent[i + 1][0] if i < len(sent) - 1 else '<END>',
    }

    features['pos'] = sent[i][1]  # Access POS 
    
    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [token[3] for token in sent] 


def prepare_dataset(dataframe):
   
    sentences = []
    current_sentence = []

    for _, row in dataframe.iterrows():
        if row[0] == '-DOCSTART-':
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
            continue

        if row.isnull().any():
            continue

        current_sentence.append(tuple(row))

    if current_sentence:
        sentences.append(current_sentence)

   
    X = [sent2features(sent) for sent in sentences]
    y = [sent2labels(sent) for sent in sentences]

    return X, y


### Train a NER Classifier Model

Implement one of the following classifiers for recognizing multiple entity types (e.g., person, organization, location): Conditional Random Field (CRF), biLSTM or multinomial logistic regression. Select only one and provide a brief explanation for
your choice of model.

In [7]:
# write your code here:

def train_crf_model(X_train, y_train):
   
    # Initialize and train CRF model
    crf = CRF(
        algorithm='lbfgs',
        c1=0.1,   # L1 regularization
        c2=0.1,   # L2 regularization
        max_iterations=100,
        all_possible_transitions=True
    )
    
    crf.fit(X_train, y_train)
    
    return crf

### Evaluation

Evaluate the model on the test set using metrics such as precision, recall, and F1-score

In [8]:
# write your code here:

def evaluate_model(crf_model, X_test, y_test):
    
    # Predict labels
    y_pred = crf_model.predict(X_test)
    
    # Flatten the predictions and true labels
    y_pred_flat = [label for sent_labels in y_pred for label in sent_labels]
    y_test_flat = [label for sent_labels in y_test for label in sent_labels]
    
    report = flat_classification_report(
        y_test, y_pred, 
        labels=['O','B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
    )
    
    return report

def main_ner_pipeline():
    # Load datasets
    train_df = pd.read_csv("ner_data/train.txt", header=0, sep=" ")
    val_df = pd.read_csv("ner_data/val.txt", header=0, sep=" ")
    test_df = pd.read_csv("ner_data/test.txt", header=0, sep=" ")
    
    # Prepare training data
    X_train, y_train = prepare_dataset(train_df)
    X_val, y_val = prepare_dataset(val_df)
    X_test, y_test = prepare_dataset(test_df)
    
    crf_model = train_crf_model(X_train, y_train)
    
    val_report = evaluate_model(crf_model, X_val, y_val)
    test_report = evaluate_model(crf_model, X_test, y_test)
    
    print("Validation Results:")
    print(val_report)
    
    print("\nTest Results:")
    print(test_report)
    
    return crf_model, val_report, test_report

crf_model, val_report, test_report = main_ner_pipeline()


C:\Users\salim\AppData\Local\Temp\ipykernel_11828\3497583199.py:68: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[0] == '-DOCSTART-':


Validation Results:
              precision    recall  f1-score   support

           O       0.99      1.00      0.99     42118
       B-PER       0.90      0.90      0.90      1842
       I-PER       0.92      0.97      0.95      1303
       B-ORG       0.88      0.81      0.85      1341
       I-ORG       0.82      0.83      0.82       751
       B-LOC       0.92      0.88      0.90      1837
       I-LOC       0.89      0.82      0.86       257
      B-MISC       0.93      0.84      0.88       922
      I-MISC       0.89      0.72      0.79       346

    accuracy                           0.98     50717
   macro avg       0.91      0.86      0.88     50717
weighted avg       0.97      0.98      0.97     50717


Test Results:
              precision    recall  f1-score   support

           O       0.98      0.99      0.99     37894
       B-PER       0.82      0.86      0.84      1617
       I-PER       0.85      0.97      0.90      1156
       B-ORG       0.82      0.70      0.76

### Reporting

Summarize your findings and suggest potential improvements for future iterations of the NER system. Additionally, discuss whether your model encountered class imbalance issues and how you addressed them. Write your suggestions to the given markdown cells.

I used these features for feature extraction : 
- token itself, token lowercase, prefix/suffix
- neighboring tokens
- pos tags, shapes

I used CRF, 

CRFs are well-suited for structured prediction tasks like NER because they model dependencies between labels in a sequence.
They excel in capturing relationships between adjacent tokens and their labels, such as ensuring valid transitions.
It is also efficient for small-scale tasks.


From the results of my implementation:

Validation Results:
High F1-scores for entities like B-PER (90%), I-PER (95%), and B-LOC (90%).
Lower F1-scores for less frequent classes like I-MISC (79%), likely due to class imbalance.

Test Results:
Similar results observed, with slightly reduced scores due to the challenge of generalizing to unseen data.

The CRF performed well on common entities like PER and LOC, achieving high precision and recall.
The high F1-scores indicate a good balance between precision and recall for most entity types.

Class imbalance was observed for I-MISC and I-LOC. (since their F1 scores are relatively low.)

To mitigate this, can augment data for underrepresented classes and use weighted loss functions during training.